# OpenSearch Specs Query Processor

This notebook connects to the OpenSearch `specs_structure` index and performs semantic search using k-NN vector similarity to find the top 15 best matches for user queries.

## Cell 1: Imports and Setup

In [38]:
import json
import os
import warnings
from typing import List, Dict, Any
from dotenv import load_dotenv
from opensearchpy import OpenSearch
from langchain_openai import OpenAIEmbeddings
from langchain_groq import ChatGroq
# Suppress SSL warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

# Configuration
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST", "localhost")
OPENSEARCH_USERNAME = os.getenv("OPENSEARCH_USERNAME", "admin")
OPENSEARCH_PASSWORD = os.getenv("OPENSEARCH_PASSWORD", "admin")
INDEX_NAME = "specs_structure"
EMBEDDING_DIMENSION = 3072

print("✓ Configuration loaded successfully")

✓ Configuration loaded successfully


## Cell 2: Helper Functions

In [39]:
def create_opensearch_client() -> OpenSearch:
    """Create and return an OpenSearch client."""
    auth = None
    if OPENSEARCH_USERNAME and OPENSEARCH_PASSWORD:
        auth = (OPENSEARCH_USERNAME, OPENSEARCH_PASSWORD)
    
    config = {
        'hosts': [OPENSEARCH_HOST],
        'http_auth': auth,
        'use_ssl': True,
        'verify_certs': False,
        'ssl_assert_hostname': False,
        'ssl_show_warn': False,
        'timeout': 3000
    }
    
    client = OpenSearch(**config)
    return client


def search_similar_specs(client: OpenSearch, query_text: str, top_k: int = 15) -> List[Dict[str, Any]]:
    """
    Perform semantic search on the specs_structure index.
    
    Args:
        client: OpenSearch client instance
        query_text: The user's search query
        top_k: Number of top results to return (default: 15)
    
    Returns:
        List of matching documents with similarity scores (only field_name and values)
    """
    # Initialize OpenAI embeddings model
    embeddings_model = OpenAIEmbeddings(
        model="text-embedding-3-large",
        openai_api_key=OPENAI_API_KEY,
        dimensions=EMBEDDING_DIMENSION
    )
    
    # Generate embedding for the query
    print(f"Generating embedding for query: '{query_text}'")
    query_embedding = embeddings_model.embed_query(query_text)
    
    # Construct k-NN search query with source filtering
    search_query = {
        "size": top_k,
        "_source": ["field_name", "values"],
        "query": {
            "knn": {
                "embedding": {
                    "vector": query_embedding,
                    "k": top_k
                }
            }
        }
    }
    
    # Execute search
    print(f"Searching for top {top_k} matches...")
    response = client.search(
        index=INDEX_NAME,
        body=search_query
    )
    
    # Parse and return results
    results = []
    for hit in response['hits']['hits']:
        result = {
            'score': hit['_score'],
            'field_name': hit['_source'].get('field_name', ''),
            'values': hit['_source'].get('values', [])
        }
        results.append(result)
    
    return results

print("✓ Helper functions defined successfully")

✓ Helper functions defined successfully


## Cell 3: Initialize Connection

In [40]:
# Create OpenSearch client
print("Connecting to OpenSearch...")
client = create_opensearch_client()

# Verify connection and check if index exists
try:
    if client.indices.exists(index=INDEX_NAME):
        index_info = client.indices.get(index=INDEX_NAME)
        doc_count = client.count(index=INDEX_NAME)['count']
        print(f"✓ Successfully connected to OpenSearch")
        print(f"✓ Index '{INDEX_NAME}' exists")
        print(f"✓ Total documents in index: {doc_count}")
    else:
        print(f"⚠ Warning: Index '{INDEX_NAME}' does not exist!")
        print("Please run create_embeddings.py first to create the index.")
except Exception as e:
    print(f"✗ Error connecting to OpenSearch: {str(e)}")

Connecting to OpenSearch...
✓ Successfully connected to OpenSearch
✓ Index 'specs_structure' exists
✓ Total documents in index: 100


## Cell 4: User Query Input

**Edit the query below to search for specifications:**

In [51]:
# Enter your search query here
user_query = "best automatic car under 12 lakh with sunroof"

print(f"Query set: '{user_query}'")

Query set: 'best automatic car under 12 lakh with sunroof'


## Cell 5: Execute Search

In [52]:
# Execute the search
try:
    results = search_similar_specs(client, user_query, top_k=10)
    print(f"✓ Search completed successfully!")
    print(f"✓ Found {len(results)} matches")
except Exception as e:
    print(f"✗ Error during search: {str(e)}")
    results = []

Generating embedding for query: 'best automatic car under 12 lakh with sunroof'
Searching for top 10 matches...
✓ Search completed successfully!
✓ Found 10 matches


## Cell 6: Display Results

In [53]:
# Display results with only Field Name and Possible Values
# Initialize object to store formatted results
formatted_output = {
    'query': user_query,
    'total_matches': 0,
    'results': []
}

if results:
    formatted_output['total_matches'] = len(results)
    
    print("="*80)
    print(f"TOP {len(results)} MATCHES FOR QUERY: '{user_query}'")
    print("="*80)
    
    for idx, result in enumerate(results, 1):
        print(f"\n{'─'*80}")
        print(f"RANK #{idx}")
        print(f"{'─'*80}")
        print(f"Field Name:        {result['field_name']}")
        print(f"Similarity Score:  {result['score']:.4f}")
        
        # Store result in formatted_output
        result_entry = {
            'field_name': result['field_name'],
            'possible_values': result['values'] if result['values'] else []
        }
        formatted_output['results'].append(result_entry)
        
        if result['values']:
            print(f"\nPossible Values:")
            for val in result['values']:
                print(f"  • {val}")
        else:
            print(f"\nPossible Values:   None")
    
    print(f"\n{'='*80}")
    print(f"END OF RESULTS")
    print(f"{'='*80}")
else:
    print("No results to display.")

print(f"\n✓ Results stored in 'formatted_output' object")

TOP 10 MATCHES FOR QUERY: 'best automatic car under 12 lakh with sunroof'

────────────────────────────────────────────────────────────────────────────────
RANK #1
────────────────────────────────────────────────────────────────────────────────
Field Name:        sunroof_moonroof
Similarity Score:  0.4830

Possible Values:
  • false
  • panoramic
  • electrically adjustable
  • fixed glass panoramic
  • electro-transparent panoramic
  • optional
  • fixed glass

────────────────────────────────────────────────────────────────────────────────
RANK #2
────────────────────────────────────────────────────────────────────────────────
Field Name:        roof_rails
Similarity Score:  0.4137

Possible Values:
  • true
  • false

────────────────────────────────────────────────────────────────────────────────
RANK #3
────────────────────────────────────────────────────────────────────────────────
Field Name:        orvm_colour
Similarity Score:  0.4095

Possible Values:
  • body-coloured
  • bl

## Cell 7: Generate API Filters using Groq

In [54]:
# Generate API filters using Groq LLM
if not results:
    print("⚠ No search results available. Please run the search first.")
elif not GROQ_API_KEY:
    print("✗ Error: GROQ_API_KEY not found in environment variables.")
    print("Please set GROQ_API_KEY in your .env file.")
else:
    try:
        print("Initializing Groq LLM...")
        
        # Setup Groq client
        groq_model = "openai/gpt-oss-120b"
        refinement_llm = ChatGroq(
            model=groq_model,
            temperature=0.0,
            api_key=GROQ_API_KEY
        )
        
        print("✓ Groq LLM initialized successfully")
        
        # Build context string from OpenSearch results
        print("Building context from search results...")
        context = "\n".join([
            f"Field: {r['field_name']}, Possible Values: {', '.join(r['values']) if r['values'] else 'None'}" 
            for r in results
        ])
        
        print("✓ Context built successfully")
        
        # Create prompt
        prompt = f"""From the user query: "{user_query}"

Interpret this into a structured JSON response that contains filters for an API call.

Reference (Field Names and Possible Values):
{context}

Instructions:
- Only add filters that are explicitly mentioned or clearly implied in the user query
- Use the reference above to map user terms to the correct field names
- Use exact values from the "Possible Values" list for each field
- Do not add filters that are not mentioned in the user query
- Output format must be: {{"filters": {{"field_name": "value", ...}}}}
- There can be multiple filters in the response
- Add min price max price filters if mentioned in the query
- There can be multiple values for a single field but it should be from the possible values list for example if user asks for sunroof then response should be "sunroof_moonroof": [panoramic, electrically adjustable, fixed, glass, panoramic, electro-transparent, panoramic, optional, fixed glass]
- Add as many values that matches user query from the possible values list
- Convert all the price values to numeric values example 15 lakh should be 1500000

Generate the JSON response:"""
        
        # Make API call to Groq
        print(f"Sending request to Groq ({groq_model})...")
        response = refinement_llm.invoke(prompt)
        
        print("✓ Received response from Groq")
        
        # Parse JSON response
        print("Parsing JSON response...")
        
        # Extract JSON from response content
        response_text = response.content.strip()
        
        # Try to find JSON in the response (in case there's extra text)
        import re
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        
        if json_match:
            json_str = json_match.group(0)
            filters_json = json.loads(json_str)
        else:
            filters_json = json.loads(response_text)
        
        print("✓ JSON parsed successfully")
        
        # Display result
        print("\n" + "="*80)
        print("GENERATED API FILTERS")
        print("="*80)
        print(f"\nQuery: '{user_query}'")
        print(f"\nFilters (Pretty-printed JSON):\n")
        print(json.dumps(filters_json, indent=2))
        print("\n" + "="*80)
        
        # Store in variable for programmatic access
        api_filters = filters_json
        print(f"\n✓ Filters stored in 'api_filters' variable")
        
    except json.JSONDecodeError as e:
        print(f"\n✗ Error: Failed to parse JSON response from Groq")
        print(f"JSON Error: {str(e)}")
        print(f"\nRaw response from Groq:")
        print(response.content)
        api_filters = None
        
    except Exception as e:
        print(f"\n✗ Error during Groq API call or processing:")
        print(f"Error type: {type(e).__name__}")
        print(f"Error message: {str(e)}")
        api_filters = None

Initializing Groq LLM...
✓ Groq LLM initialized successfully
Building context from search results...
✓ Context built successfully
Sending request to Groq (openai/gpt-oss-120b)...
✓ Received response from Groq
Parsing JSON response...
✓ JSON parsed successfully

GENERATED API FILTERS

Query: 'best automatic car under 12 lakh with sunroof'

Filters (Pretty-printed JSON):

{
  "filters": {
    "sunroof_moonroof": [
      "panoramic",
      "electrically adjustable",
      "fixed glass panoramic",
      "electro-transparent panoramic",
      "optional",
      "fixed glass"
    ],
    "transmission_type": [
      "automatic",
      "automatic (tc)",
      "automatic (dct)",
      "automatic (amt)",
      "automatic (cvt)",
      "automatic (e-cvt)"
    ],
    "max_price": 1200000
  }
}


✓ Filters stored in 'api_filters' variable
